In [1]:
import numpy as np

# System Parameters
V_NOMINAL = 588  # Nominal battery voltage (V)
NUM_RESISTORS = 5  # Resistors per port bank
DERATING_FACTOR = 0.5  # 2x safety

# Standard power ratings (W)
STANDARD_WATTAGE_OPTIONS = np.array([0.25, 0.5, 1.0, 2.0, 3.0, 5.0])

# Test currents in Amps (1.0 mA to 3.0 mA)
I_TEST = np.array([1.0, 1.5, 2.0, 2.5, 3.0]) / 1000  # A
I_TRIP_MIN = 2.0 / 1000  # 2.0 mA IMD trip threshold target

# Resistor vals:
R_ind_standard = np.array([120.0, 82.0, 56.0, 47.0, 39.0]) * 1000  # Ω

# Actual Calculations
R_total_actual = R_ind_standard * NUM_RESISTORS
I_actual = V_NOMINAL / R_total_actual
P_total_actual = (I_actual**2) * R_total_actual
P_ind_actual = P_total_actual / NUM_RESISTORS  # Power dissipation per resistor

# MAx power dissipation across all ports
max_p_per_resistor = np.max(P_ind_actual)

# Minimum rating needed after applying
min_required_rating_w = max_p_per_resistor / DERATING_FACTOR

# Smallest standard component wattage that covers the requirement
recommended_rating_w = STANDARD_WATTAGE_OPTIONS[
    STANDARD_WATTAGE_OPTIONS >= min_required_rating_w
][0]

# Voltage Calculations
V_per_resistor = V_NOMINAL / NUM_RESISTORS

# Voltage limits by package type
VOLTAGE_LIMITS = {
    "1206 SMD": 200.0,
    "2512 SMD": 300.0,
    "1W Through-Hole Metal Film": 350.0,
    "2W+ High Voltage Metal Film": 500.0,
}

# --- Display Output ---
print("=== DYNAMIC VOLTAGE & ARCING ANALYSIS ===")
print(f"System Voltage:                     {V_NOMINAL:.1f} V")
print(f"Voltage Drop per Resistor (5x):     {V_per_resistor:.1f} V/resistor\n")

print("Working Voltage Limits by Package:")
for pkg, limit in VOLTAGE_LIMITS.items():
    status = "SAFE" if V_per_resistor <= limit else "EXCEEDS LIMIT"
    print(f"  - {pkg:<28} (Max {limit:.0f}V): {status}")

print("\n=== DYNAMIC POWER & SIZING SELECTION ===")
print(f"Max Thermal Stress per Resistor:    {max_p_per_resistor:.3f} W (Port 5)")
print(
    f"Required Rating (@ 50% Derating):   {min_required_rating_w:.3f} W minimum"
)
print(f"--> RECOMMENDED UNIFIED RATING:     {recommended_rating_w:.1f} W\n")

print("=== PORT SPECIFICATIONS (E24 STANDARD VALUES) ===")
for i in range(len(I_TEST)):
    load_pct = (P_ind_actual[i] / recommended_rating_w) * 100
    print(
        f"Port {i+1}: 5x {R_ind_standard[i]/1000:.1f} kΩ ({recommended_rating_w:.1f}W) | "
        f"Actual Current: {I_actual[i]*1000:.2f} mA | "
        f"Power/Resistor: {P_ind_actual[i]:.3f} W ({load_pct:.1f}% load)"
    )

=== DYNAMIC VOLTAGE & ARCING ANALYSIS ===
System Voltage:                     588.0 V
Voltage Drop per Resistor (5x):     117.6 V/resistor

Working Voltage Limits by Package:
  - 1206 SMD                     (Max 200V): SAFE
  - 2512 SMD                     (Max 300V): SAFE
  - 1W Through-Hole Metal Film   (Max 350V): SAFE
  - 2W+ High Voltage Metal Film  (Max 500V): SAFE

=== DYNAMIC POWER & SIZING SELECTION ===
Max Thermal Stress per Resistor:    0.355 W (Port 5)
Required Rating (@ 50% Derating):   0.709 W minimum
--> RECOMMENDED UNIFIED RATING:     1.0 W

=== PORT SPECIFICATIONS (E24 STANDARD VALUES) ===
Port 1: 5x 120.0 kΩ (1.0W) | Actual Current: 0.98 mA | Power/Resistor: 0.115 W (11.5% load)
Port 2: 5x 82.0 kΩ (1.0W) | Actual Current: 1.43 mA | Power/Resistor: 0.169 W (16.9% load)
Port 3: 5x 56.0 kΩ (1.0W) | Actual Current: 2.10 mA | Power/Resistor: 0.247 W (24.7% load)
Port 4: 5x 47.0 kΩ (1.0W) | Actual Current: 2.50 mA | Power/Resistor: 0.294 W (29.4% load)
Port 5: 5x 39.0 kΩ (